# Notebook 05 — Evaluation and Paper Figures

Loads baseline scores, computes KCS-DT + GGS, runs statistical tests, and produces all figures for Paper 2.

**No GPU required.**

In [ ]:
import os, sys, json
REPO = 'ifc-graphrag-dt'
if os.path.exists(REPO): !cd {REPO} && git pull --quiet
else: !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
sys.path.insert(0, REPO)

!pip install scipy scikit-learn seaborn --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import defaultdict

from evaluation.metrics.kcs_dt import KCSDTScorer
from evaluation.metrics.ggs import GGSScorer
from evaluation.results.statistical_tests import (
    wilcoxon_signed_rank, cohen_d, bootstrap_ci, run_full_analysis
)
from benchmark.annotation_tool.iaa_compute import compute_iaa

SCORES_DIR  = f'{REPO}/outputs/scores'
FIGURES_DIR = f'{REPO}/outputs/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

BASELINE_NAMES = {
    'b0': 'B0: No Grounding',
    'b1': 'B1: Prompt LLM',
    'b2': 'B2: Flat RAG',
    'b3': 'B3: IFC Lookup',
    'b4': 'B4: GraphRAG-DT'
}
COLORS = {'b0':'#888780','b1':'#B4B2A9','b2':'#5DCAA5','b3':'#EF9F27','b4':'#2E5FA3'}
print('Imports OK')

In [ ]:
# ── Cell 2: Load all scores ───────────────────────────────────────────────────
all_scores = {}
for b in ['b0','b1','b2','b3','b4']:
    path = Path(f'{SCORES_DIR}/{b}_scores.json')
    if path.exists():
        with open(path) as f:
            all_scores[b] = json.load(f)
        print(f'{b}: {len(all_scores[b])} records')
    else:
        print(f'{b}: NOT FOUND — run Notebook 04 first')
        all_scores[b] = []

In [ ]:
# ── Cell 3: Compute KCS-DT scores per baseline ────────────────────────────────
# For each prompt, we compute KCS-DT using spec entity/relation counts
# as a proxy until full ground-truth annotations are available.
# With GT annotations, replace this cell with annotated Stage B scores.

scorer = KCSDTScorer()

def compute_kcs_proxy(records):
    """Compute proxy KCS-DT from entity/relation counts (pre-annotation)."""
    scores = []
    for r in records:
        if r.get('status') != 'ok':
            continue
        n_entities  = r.get('entities', 0)
        n_relations = r.get('relations', 0)
        tier = r.get('tier', 1)
        # Proxy: normalise against expected entity/relation counts per tier
        expected_entities  = {1: 1, 2: 3, 3: 7}.get(tier, 3)
        expected_relations = {1: 0, 2: 2, 3: 5}.get(tier, 2)

        e  = min(n_entities  / expected_entities,  1.0) if expected_entities  > 0 else 1.0
        r_ = min(n_relations / expected_relations, 1.0) if expected_relations > 0 else 1.0

        kcs = 0.20*e + 0.35*r_ + 0.15*1.0 + 0.15*1.0 + 0.15*1.0  # A, Cn, Cv default 1.0 for proxy
        scores.append({'prompt_id': r['prompt_id'], 'tier': tier, 'total': round(kcs, 4),
                        'entity': e, 'relation': r_, 'baseline': r.get('baseline','')})
    return scores

kcs_scores = {b: compute_kcs_proxy(recs) for b, recs in all_scores.items()}

print('KCS-DT proxy scores computed (replace with annotated scores after expert annotation).')
for b, scores in kcs_scores.items():
    if scores:
        print(f'  {b}: mean={np.mean([s["total"] for s in scores]):.3f} '
              f'(n={len(scores)})')

In [ ]:
# ── Cell 4: Main result table (Table 1 of Paper 2) ────────────────────────────
print('TABLE 1: KCS-DT by Baseline and Tier')
print('='*68)
print(f'{"Baseline":<22} {"Tier 1 (Asset)":>16} {"Tier 2 (Assembly)":>18} {"Tier 3 (System)":>16}')
print('-'*68)

table_data = {}
for b in ['b0','b1','b2','b3','b4']:
    row = []
    for tier in [1, 2, 3]:
        tier_scores = [s['total'] for s in kcs_scores.get(b,[]) if s['tier']==tier]
        if tier_scores:
            mean = np.mean(tier_scores)
            std  = np.std(tier_scores)
            row.append((mean, std, tier_scores))
            cell = f'{mean:.3f} ± {std:.3f}'
        else:
            row.append((0,0,[]))
            cell = 'N/A'
        print(f'{BASELINE_NAMES.get(b,b):<22}', end='') if tier==1 else None
        print(f' {cell:>17}', end='')
    print()
    table_data[b] = row

print('='*68)
print('Note: These are proxy scores. Replace with expert-annotated KCS-DT after Stage B annotation.')

In [ ]:
# ── Cell 5: Figure 1 — KCS-DT grouped bar chart ───────────────────────────────
tiers = [1, 2, 3]
tier_labels = ['Tier 1\n(Asset)', 'Tier 2\n(Assembly)', 'Tier 3\n(System)']
baselines = ['b0','b1','b2','b3','b4']
n_baselines = len(baselines)
x = np.arange(len(tiers))
width = 0.15

fig, ax = plt.subplots(figsize=(12, 6))

for i, b in enumerate(baselines):
    means = [table_data[b][t][0] for t in range(3)]
    stds  = [table_data[b][t][1] for t in range(3)]
    offset = (i - n_baselines/2 + 0.5) * width
    bars = ax.bar(x + offset, means, width, label=BASELINE_NAMES[b],
                  color=COLORS[b], yerr=stds, capsize=3,
                  edgecolor='white', linewidth=0.5)

ax.set_xlabel('DTAH-Bench Tier', fontsize=12)
ax.set_ylabel('KCS-DT Score', fontsize=12)
ax.set_title('IFC-GraphRAG-DT vs Baselines — KCS-DT by Tier (DTAH-Bench-50)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(tier_labels, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(y=0.73, color='#2E5FA3', linestyle='--', alpha=0.5, label='H1 target (0.73)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_kcs_dt_grouped_bar.png', dpi=200)
plt.show()
print(f'Figure saved: {FIGURES_DIR}/05_kcs_dt_grouped_bar.png')

In [ ]:
# ── Cell 6: Figure 2 — KCS-DT sub-score radar chart (B2 vs B4) ───────────────
import matplotlib.patches as patches

# Sample sub-score data from scores
subscore_names = ['Entity', 'Relation', 'Attribute', 'Containment', 'Connectivity']
n_scores = len(subscore_names)

def get_mean_subscores(b, tier=3):
    data = [s for s in kcs_scores.get(b,[]) if s['tier']==tier]
    if not data: return [0]*5
    return [
        np.mean([s.get('entity',0) for s in data]),
        np.mean([s.get('relation',0) for s in data]),
        1.0, 1.0, 1.0  # A, Cn, Cv are proxy 1.0
    ]

b2_scores = get_mean_subscores('b2', tier=3)
b4_scores = get_mean_subscores('b4', tier=3)

angles = np.linspace(0, 2*np.pi, n_scores, endpoint=False).tolist()
b2_scores += [b2_scores[0]]
b4_scores += [b4_scores[0]]
angles += [angles[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
ax.plot(angles, b2_scores, 'o-', color='#5DCAA5', linewidth=2, label='B2: Flat RAG')
ax.fill(angles, b2_scores, alpha=0.15, color='#5DCAA5')
ax.plot(angles, b4_scores, 'o-', color='#2E5FA3', linewidth=2, label='B4: GraphRAG-DT')
ax.fill(angles, b4_scores, alpha=0.15, color='#2E5FA3')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(subscore_names, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('KCS-DT Sub-score Profile\nB2 (Flat RAG) vs B4 (GraphRAG-DT) — Tier 3', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_kcs_dt_radar.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: Figure 3 — Error taxonomy stacked bar ────────────────────────────
# Placeholder with realistic error distributions based on prior work
# Replace with actual error codes after annotation

error_categories = ['EA-1\nEntity miss', 'EA-2\nRelation miss', 'EA-3\nMulti-hop fail',
                    'EA-4\nContainment miss', 'EA-5\nAttribute err']

# Error counts per baseline (Tier 3, n=20 prompts)
# These are realistic estimates — replace with actual from annotation
error_data = {
    'B0: No Grounding': [18, 20, 20, 19, 15],
    'B1: Prompt LLM':   [12, 16, 18, 14, 10],
    'B2: Flat RAG':     [ 9, 17, 18, 13,  8],
    'B3: IFC Lookup':   [ 5, 11, 14,  9,  6],
    'B4: GraphRAG-DT':  [ 2,  5,  6,  4,  3],
}

x = np.arange(len(error_categories))
fig, ax = plt.subplots(figsize=(12, 6))

colors_err = ['#888780','#B4B2A9','#5DCAA5','#EF9F27','#2E5FA3']
bottom = np.zeros(len(error_categories))
for (label, counts), color in zip(error_data.items(), colors_err):
    ax.bar(x, counts, label=label, color=color, bottom=bottom, edgecolor='white')
    bottom += np.array(counts)

ax.set_xticks(x)
ax.set_xticklabels(error_categories, fontsize=10)
ax.set_ylabel('Error Count (Tier 3, n=20 prompts)', fontsize=11)
ax.set_title('Stage A Error Taxonomy by Baseline — Tier 3 System Prompts', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_error_taxonomy_stacked.png', dpi=200)
plt.show()
print('Note: Replace error_data with actual annotation counts from Stage A evaluation.')

In [ ]:
# ── Cell 8: Statistical tests ─────────────────────────────────────────────────
print('STATISTICAL ANALYSIS')
print('=' * 60)

# Save scores in the format expected by run_full_analysis
for b, scores in kcs_scores.items():
    out = f'{SCORES_DIR}/{b}_scores.json'
    # Merge with existing records
    existing = []
    if Path(out).exists():
        with open(out) as f:
            existing = json.load(f)
    # Add KCS-DT total to each record
    scores_by_id = {s['prompt_id']: s for s in scores}
    for rec in existing:
        pid = rec.get('prompt_id','')
        if pid in scores_by_id:
            rec['total'] = scores_by_id[pid]['total']
    with open(out, 'w') as f:
        json.dump(existing, f, indent=2)

report = run_full_analysis(
    results_dir=SCORES_DIR,
    output_path=f'{REPO}/outputs/results/statistical_report.json'
)

# Print primary comparison
pc = report.get('primary_comparison', {})
if pc:
    print('\nPrimary comparison: B4 (GraphRAG-DT) vs B2 (Flat RAG) at Tier 3')
    wlx = pc.get('wilcoxon', {})
    eff = pc.get('effect_size', {})
    b4ci = pc.get('b4_ci', {})
    b2ci = pc.get('b2_ci', {})
    print(f'  B4 mean KCS-DT: {b4ci.get("mean","N/A")} [{b4ci.get("ci_lower","")}, {b4ci.get("ci_upper","")}]')
    print(f'  B2 mean KCS-DT: {b2ci.get("mean","N/A")} [{b2ci.get("ci_lower","")}, {b2ci.get("ci_upper","")}]')
    print(f'  Wilcoxon p-value: {wlx.get("p_value","N/A")} (significant: {wlx.get("significant",False)})')
    print(f'  Cohen\'s d: {eff.get("cohen_d","N/A")} ({eff.get("magnitude","N/A")} effect)')
else:
    print('Primary comparison: insufficient data (need scores from Notebook 04)')

In [ ]:
# ── Cell 9: Figure 4 — GGS heatmap (Flat RAG vs GraphRAG by tier) ─────────────
# Load GGS data computed in Notebook 02
# For demonstration, use realistic GGS values

# GGS = [NodeRecall, EdgeRecall, PathRecall, Total]
ggs_data = {
    'Flat RAG': {
        'Tier 1': [0.88, 0.00, 0.00, 0.31],
        'Tier 2': [0.82, 0.09, 0.04, 0.33],
        'Tier 3': [0.74, 0.07, 0.02, 0.29],
    },
    'GraphRAG-DT': {
        'Tier 1': [0.90, 0.88, 0.82, 0.88],
        'Tier 2': [0.88, 0.79, 0.71, 0.81],
        'Tier 3': [0.83, 0.74, 0.62, 0.74],
    }
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ggs_cols = ['Node Recall', 'Edge Recall', 'Path Recall', 'GGS Total']
tier_rows = ['Tier 1', 'Tier 2', 'Tier 3']

for ax, (method, data) in zip(axes, ggs_data.items()):
    matrix = np.array([data[t] for t in tier_rows])
    im = ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(4))
    ax.set_xticklabels(ggs_cols, fontsize=10, rotation=15)
    ax.set_yticks(range(3))
    ax.set_yticklabels(tier_rows, fontsize=10)
    ax.set_title(f'GGS — {method}', fontsize=12)
    for i in range(3):
        for j in range(4):
            ax.text(j, i, f'{matrix[i,j]:.2f}',
                    ha='center', va='center', fontsize=11,
                    color='white' if matrix[i,j] > 0.6 else 'black')
    plt.colorbar(im, ax=ax)

plt.suptitle('Graph Grounding Score (GGS): Flat RAG vs GraphRAG-DT', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_ggs_heatmap.png', dpi=200)
plt.show()
print('Replace heatmap data with actual GGS scores from Notebook 02 ground-truth traversal.')

In [ ]:
# ── Cell 10: Export all figures to Drive ──────────────────────────────────────
import shutil
try:
    drive_figs = '/content/drive/MyDrive/ifc_graphrag_dt_outputs/figures'
    shutil.copytree(FIGURES_DIR, drive_figs, dirs_exist_ok=True)
    print(f'All figures saved to Google Drive: {drive_figs}')
except Exception:
    print(f'Figures saved locally: {FIGURES_DIR}')

import os
figs = sorted(os.listdir(FIGURES_DIR))
print(f'\nFigures produced ({len(figs)} total):')
for fig in figs:
    size = os.path.getsize(f'{FIGURES_DIR}/{fig}')
    print(f'  {fig} ({size/1024:.1f} KB)')